# THERMA SEARCH AI · 03 · WiSARD tek çalıştırmada analiz

**CPU çalışma zamanı → Tümünü çalıştır.** Drive ilk bağlantıda Google’ın erişim iznini isteyebilir; bu Colab gereğidir. Analiz için hücre, regex, kare numarası veya şema girmeniz gerekmez.

Mevcut arşiv: `/content/drive/MyDrive/ThermaSearchAI/datasets/WiSARD_Multi_Modal_Sample.zip`.
Bütün raporlar yeni bir `/content/drive/MyDrive/ThermaSearchAI/reports/wisard/run_wisard_.../` klasörüne otomatik kaydedilir.

[WiSARD resmî sayfası](https://sites.google.com/uw.edu/wisard/) · [IROS 2022 makalesi](https://doi.org/10.1109/IROS47612.2022.9981298).
Kullanıcının gerçek örnek incelemesi bilinen IR/VIS klasörlerini, normalize YOLO etiketlerini ve sınıf 0=Person eşlemesini doğrulamıştır. Önceki 264/264 görüntü, 1006/1022 anotasyon sayıları **referanstır**; kod bunları bağımsız yeniden hesaplar. Tam arşiv indirilmez, eğitim/split oluşturulmaz, görüntüler ve etiketler değiştirilmez.

**Eşleştirme:** A: IR n ↔ VIS n; B: IR n ↔ VIS n−1. İkisi de analiz edilir. Sayım uyumu senkronizasyon kanıtı değildir. Kaynak beyanı, yerel ölçüm ve geçici hipotez raporda ayrılır. Kırmızı kutular **ground truth** anotasyonlarıdır; model tahmini değildir.

## 1 · Kod kurulumu
Public proje kodu sabit revizyondan yüklenir. Yalnızca Pillow/Matplotlib kontrol edilir; Colab ipykernel/jupyter-client paketleri değiştirilmez. Veri seti indirilmez.

In [ ]:
from pathlib import Path
import os
import sys
import subprocess
import json

REPOSITORY = 'https://github.com/hilalaltunayy/therma-search-ai.git'
CODE_REF = 'feature/wisard-one-pass'  # Teslimde sınanan yardımcı kod commit’ine sabitlenir.
local_code_root = os.environ.get('THERMA_CODE_ROOT')
if local_code_root:
    PROJECT_ROOT = Path(local_code_root).resolve()
else:
    PROJECT_ROOT = Path('/content/therma-search-ai-wisard')
    if not PROJECT_ROOT.exists():
        subprocess.run(['git','clone','--quiet',REPOSITORY,str(PROJECT_ROOT)],check=True)
    remote = subprocess.check_output(['git','-C',str(PROJECT_ROOT),'remote','get-url','origin'],text=True).strip()
    if remote != REPOSITORY:
        raise RuntimeError('Mevcut kod klasörünün kaynağı beklenen depo değil.')
    if subprocess.check_output(['git','-C',str(PROJECT_ROOT),'status','--porcelain'],text=True).strip():
        raise RuntimeError('Yerel kod değişikliği var; üzerine yazılmadı. Temiz çalışma zamanı kullanın.')
    subprocess.run(['git','-C',str(PROJECT_ROOT),'fetch','--quiet','--depth','1','origin',CODE_REF],check=True)
    subprocess.run(['git','-C',str(PROJECT_ROOT),'checkout','--quiet','--detach','FETCH_HEAD'],check=True)
    subprocess.run([sys.executable,'-m','pip','install','--quiet','--no-deps','-r',str(PROJECT_ROOT/'requirements-colab.txt')],check=True)
sys.path.insert(0,str(PROJECT_ROOT))
for module in list(sys.modules):
    if module == 'src' or module.startswith('src.'):
        del sys.modules[module]
CODE_REVISION = subprocess.check_output(['git','-C',str(PROJECT_ROOT),'rev-parse','HEAD'],text=True).strip()
from src.data.wisard_reports import run_wisard_sample_analysis
from src.data.wisard_sample_reports import turkish_summary
from IPython.display import display, Image as DisplayImage
print('Kod commit:', CODE_REVISION)


## 2 · Drive ve doğrulanmış örnek yolu
Varsayılan yol hazırdır. ZIP yoksa indirme yapılmaz; açık tanılama raporu oluşturulur. Bilinmeyen yapı gelirse envanter çıkarılır, etiket/eşleştirme tahmin edilmez. 2 GiB ZIP, 8 GiB toplam üye, 32 MiB üye ve 25 milyon piksel sınırları korunur.

In [ ]:
if not local_code_root:
    from google.colab import drive
    drive.mount('/content/drive')
DATA_DIR = Path(os.environ.get('THERMA_WISARD_DATA_DIR', '/content/drive/MyDrive/ThermaSearchAI/datasets'))
REPORT_ROOT = Path(os.environ.get('THERMA_WISARD_REPORT_ROOT', '/content/drive/MyDrive/ThermaSearchAI/reports/wisard'))
# Ortam değişkenleri yalnızca yerel/sentetik doğrulama içindir; Colab’da ayar gerekmez.
archive_override = os.environ.get('THERMA_WISARD_ZIP')
SAMPLE_ZIP_PATH = (Path(archive_override) if archive_override else None) if archive_override is not None else DATA_DIR/'WiSARD_Multi_Modal_Sample.zip'
print('Arşiv:', SAMPLE_ZIP_PATH)
print('Otomatik kayıt:', REPORT_ROOT)

## 3 · Bütünlük, anotasyon, eşleştirme, sızıntı ve raporlar
Tüm görüntüler gerçekten çözülür; etiketler, sınıf ID’leri ve kutu sınırları denetlenir. IR/VIS ayrı sayılır. `count.txt` etiket sayılmaz. Boş anotasyon otomatik negatif sayılmaz.

Her iki hipotez için kapsam, sayım farkı dağılımları ve ardışık değişimler üretilir. Yeterli okunabilir aday varsa her hipotezden **10 farklı kare**, kendi piksel uzayındaki ground-truth kutularıyla kaydedilir. dHash/SHA ve nesne konumundaki değişimler yalnızca **aynı modalite içinde** incelenir. Metadata ve EXIF zaman alanları kaydedilir; ZIP dosya tarihi kamera zamanı kabul edilmez. Tek uçuş örneği bağımsız uçuş değerlendirmesi sağlamaz.

PNG’ler 200 DPI; JSON/CSV ve 10 bölümlü akademik Markdown raporu otomatik kaydedilir. Bu hücre tüm analizdir; veri boyutuna ve Drive hızına göre birkaç dakika sürebilir.

In [ ]:
WISARD_RUN_DIR, WISARD_SUMMARY = run_wisard_sample_analysis(
    REPORT_ROOT, archive_path=SAMPLE_ZIP_PATH, code_revision=CODE_REVISION,
)
print('Rapor üretimi:', WISARD_SUMMARY['status'])

## 4 · Kaydedilen grafiklerden önizleme
Bütün PNG’ler ve her iki hipotezin tüm görselleri Markdown raporundadır. Aşağıda sınırlı önizleme gösterilir; dosyalar Drive’a zaten kaydedilmiştir.

In [ ]:
preview_names = ['class_distribution.png', 'people_per_frame.png', 'bbox_width_height.png',
                 'small_person_statistics.png', 'pairing_count_differences.png',
                 'temporal_similarity_and_positions.png', 'empty_annotations_01.png',
                 'pair_A_same_number_01.png', 'pair_B_shift_minus_one_01.png']
for name in preview_names:
    path = WISARD_RUN_DIR/name
    if path.exists():
        display(DisplayImage(filename=str(path), width=1000))
print('Markdown rapor:', WISARD_RUN_DIR/'wisard_analysis_report.md')
print('Dosya sayısı:', len(list(WISARD_RUN_DIR.iterdir())))

## 5 · Türkçe sonuç özeti
Anotasyon sayıları benzersiz kişi sayıları değildir. Geçici eşleştirme tercihi fiziksel senkronizasyonu kesinleştirmez. Boş etiketler, lisans/sürüm ve bağımsız uçuş genellemesi ayrıca doğrulanmayı bekler.

In [ ]:
print(turkish_summary(WISARD_RUN_DIR, WISARD_SUMMARY))